In [1]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

BASE_DIR = Path.cwd().parent
DB_PATH = BASE_DIR/ 'database'/ 'olist.db'

engine = create_engine(f'sqlite:///{DB_PATH}')

In [2]:
# Load the full orders table into pandas for cleaning
# From this point we work in pandas, not SQL
# SQL already did its job in notebook 02 by experimenting with queiries and save to file

orders = pd.read_sql("SELECT * FROM orders", engine)

print(f"Total rows:{len(orders):,}")
print(f"Total columns:{len(orders.columns)}")
print(f"\nColumn names:")
print(list(orders.columns))
orders.head(2)

Total rows:99,441
Total columns:8

Column names:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00


In [3]:
# isnull().sum() counts how many NULL/NaN per column
# divide by total rows x 100 = percentage missing

orders_missing = pd.DataFrame({
    'missing_count'  : orders.isnull().sum(),
    'missing_percent': (orders.isnull().sum() / len(orders) * 100).round(2)
})

#Descending sort by missing_count to see whcih columns have the most missing values
missing_sorted = orders_missing.sort_values(by='missing_count', ascending=False)

# only show columns that actually have missing values so missing_count > 0
print("Missing values (sorted):")
missing_sorted[missing_sorted['missing_count'] > 0]



Missing values (sorted):


,missing_count,missing_percent
order_delivered_customer_date,2965,2.98
order_delivered_carrier_date,1783,1.79
order_approved_at,160,0.16


In [4]:
print("Current data info:")
print(orders.info())

Current data info:
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB
None


In [5]:
# Convert the timestamp columns in orders df to datetime format
# e.g. "2017-10-02 10:56:33" — pandas doesn't know this is a date yet
# so we need to convert them to datetime format for easier analysis later

timestamp_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in timestamp_cols:
    orders[col] = pd.to_datetime(
        orders[col],
        errors='coerce'  # if a value can't be converted, set to NaT instead of crashing
    )
#NaT = Not a Time
#It’s the datetime version of NaN

print("Updated data types:")
print(orders[timestamp_cols].dtypes)
# should now show datetime64 instead of object

# print again orders df info to see all columns data types
print("\n all columns info):")
print(orders.info())

Updated data types:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

 all columns info):
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   

In [6]:
# Problem: some orders marked 'delivered' but have no delivery timestamp
# but if delivered, must have a delivery date
# maybe a dataset quality issue

impossible = orders[
    (orders['order_status'] == 'delivered') &
    (orders['order_delivered_customer_date'].isnull())
]

print(f"Orders marked 'delivered' but missing delivery date: {len(impossible)} orders")
print("\nThose orders rows without delivery dates:")
impossible[['order_id', 'order_status',
            'order_purchase_timestamp',
            'order_delivered_customer_date']].head(8) # since the problem orders is only 8 rows,we can show them all

Orders marked 'delivered' but missing delivery date: 8 orders

Those orders rows without delivery dates:


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,NaT
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,NaT
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,NaT
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,NaT
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,NaT
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,NaT
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,NaT
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,NaT


In [7]:
duplic = orders.duplicated(subset='order_id').sum()
#Check duplicate order_id is True/False
#.sum() helps count how many True(duplicates)there are,if no sum,its will show all rows with True/False
print(f"Duplicate order_ids: {duplic}")
# Check again in pandas cleaning steps
# expect 0 since PRIMARY KEY in schema.sql already prevented duplicates at insert

Duplicate order_ids: 0


In [8]:
# Create the new columns we want to add to the cleanned dataset
# Calculate actual delivery days — how long from purchase to delivery
orders['actual_delivery_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_purchase_timestamp']
).dt.days
# .dt.days extracts the integer number of days from a time difference
# .dt.days help convert datetime64 to integer(int64) 1 2 3 days insted of show as 2017-10-02 10:56:33(like order_delibeed or order_purchase columns)

# Calculate early or late vs estimated date
# positive = arrived BEFORE estimated date (early)
# negative = arrived AFTER estimated date (late)
orders['early_or_late_days'] = (
    orders['order_estimated_delivery_date'] -
    orders['order_delivered_customer_date']
).dt.days

print("New columns added:")
orders[['order_id',
        'actual_delivery_days',
        'early_or_late_days']].dropna().head(10)

New columns added:


,order_id,actual_delivery_days,early_or_late_days
0,e481f51cbdc54678b7cc49136f2d6af7,8.0,7.0
1,53cdb2fc8bc7dce0b6741e2150273451,13.0,5.0
2,47770eb9100c2d0c44946d9cf07ec65d,9.0,17.0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0,12.0
4,ad21c59c0840e6cb83a9ceb5573f8159,2.0,9.0
5,a4591c265e18cb1dcee52889e2d8acc3,16.0,5.0
7,6514b8ad8028c9f2cc2374ded245783f,9.0,11.0
8,76c6e866289321a7c93b82b54852dc33,9.0,31.0
9,e69bfb5eb88e0ed6a785585b27e16dbf,18.0,6.0
10,e6ce16cb79ec1d90b1da9085a6118aeb,12.0,8.0


In [9]:
# Save as a NEW table called orders_clean
# Never overwrite the original orders table — raw data stays untouched
# orders_clean has everything orders has + 2 new derived columns

orders.to_sql(
    'orders_clean', #new table in database
    engine,
    if_exists='replace',  # replace is fine here, orders_clean is derived not raw
    index=False
)

print(f" orders_clean saved — {len(orders):,} rows")
print(f" New columns: actual_delivery_days, early_or_late_days")

 orders_clean saved — 99,441 rows
 New columns: actual_delivery_days, early_or_late_days


In [10]:
# Confirm orders_clean exists in database with correct row count
# Remove impossible orders for has_delivery_days with missing delivery dates by using actual_delivery_days IS NOT NULL
verify = pd.read_sql("""
    SELECT
        COUNT(*)                                    AS total_rows,
        COUNT(actual_delivery_days)                 AS has_delivery_days,
        ROUND(AVG(actual_delivery_days), 1)         AS avg_actual_delivery_days,
        ROUND(MIN(actual_delivery_days), 0)         AS min_days,
        ROUND(MAX(actual_delivery_days), 0)         AS max_days
    FROM orders_clean
    WHERE actual_delivery_days IS NOT NULL
""", engine)

verify

,total_rows,has_delivery_days,avg_actual_delivery_days,min_days,max_days
0,96476,96476,12.1,0.0,209.0
